[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap06/cap06.EPs_aluno.ipynb)
[![](imagens/github-badge.png){width="19%"}](https://github.com/fzampirolli/pdi-vc)


## 💻 Parte Prática com Exercícios de Programação

🚧 **Em construção!**

Os exercícios de programação (EP) desta seção complementam os conceitos apresentados ao longo do Capítulo 6 por meio da implementação de algoritmos relacionados à inspeção industrial e à análise de documentos. O objetivo é consolidar os fundamentos estudados, reproduzindo, em escala reduzida, etapas de um *pipeline* típico de Visão Computacional.

Diferentemente dos capítulos anteriores, cujos exercícios enfatizavam operações mais diretamente relacionadas aos dados de imagem, os EPs deste capítulo concentram-se nas **grandezas intermediárias** produzidas durante o processamento, como áreas, perímetros, circularidade, ângulos de retas, graus de preenchimento de bolhas, mapas de variância e mapas de diferença. Essa abordagem permite compreender e validar cada etapa do *pipeline* de forma independente, sem depender de bibliotecas especializadas para aquisição de imagens, detecção de marcadores ou decodificação de códigos.

Os exercícios seguem a mesma sequência conceitual do capítulo, em ordem crescente de complexidade. Inicialmente, são abordadas métricas de avaliação de segmentação, utilizadas para quantificar a qualidade de máscaras binárias. Em seguida, estudam-se critérios geométricos para seleção de marcadores, classificação de marcações em formulários e estimação da inclinação de documentos por meio da Transformada de Hough. Na parte final, os exercícios exploram a normalização de iluminação, a detecção de defeitos por análise de textura e a integração entre registro geométrico e subtração de imagens em um *pipeline* simplificado de inspeção industrial.

Cada exercício representa uma etapa isolada de um sistema real de Visão Computacional, permitindo validar individualmente conceitos que, em aplicações industriais, são combinados em um único *pipeline* de inspeção.

### 🗺️ Legenda de Dificuldade {.unnumbered}

| Nível | Significado | EPs |
|:---:|---|---|
| 🟢 | Muito fácil / fácil — implementação de um único conceito ou algoritmo simples | EP06_01, EP06_02 |
| 🟡 | Fácil–médio — tratamento de múltiplos casos ou utilização de critérios estatísticos simples | EP06_03, EP06_04 |
| 🟠 | Médio — processamento matricial ponto a ponto | EP06_05 |
| 🔴 | Difícil — processamento matricial com operações em vizinhança (janela deslizante) | EP06_06 |
| 🟣 | Muito difícil — integração de múltiplas etapas de um *pipeline* de Visão Computacional | EP06_07 |

::: {.callout-important}
### Diretrizes para a Resolução dos Exercícios de Programação {.unnumbered}

Salvo indicação em contrário, todos os exercícios utilizam a convenção de coordenadas matriciais `[linha][coluna]`, com origem em $(0,0)$ no canto superior esquerdo da imagem.

Quando houver necessidade de arredondamento numérico, deve-se utilizar o arredondamento padrão para o inteiro mais próximo (*round half away from zero*, com `np.floor(img + 0.5)`). Comparações com limiares (por exemplo, circularidade, variância, diferença de intensidade ou grau de preenchimento) devem ser consideradas **estritas** (`>`), exceto quando o enunciado especificar explicitamente outro critério.

Cada exercício foi elaborado para enfatizar um conceito específico apresentado no capítulo. Recomenda-se implementar inicialmente a solução de forma direta e, somente após sua validação, buscar alternativas mais eficientes ou mais gerais.
:::

### 🎯 Objetivo deste Caderno {.unnumbered}

Este caderno foi elaborado para apoiar o desenvolvimento, a validação e os testes das soluções dos **Exercícios de Programação (EPs)** em um ambiente interativo, como o Google Colab ou o Jupyter Notebook. Após verificar o funcionamento da implementação com os casos de teste apresentados, o código pode ser submetido ao Moodle para a avaliação oficial.

#### *Download* {.unnumbered}

Execute a célula a seguir para obter os arquivos `morph.py` e `testsuite.py`, utilizados pelos exercícios deste capítulo.

In [105]:
import os, sys, importlib, inspect, urllib.request

# URLs do repositório
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py", "testsuite.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

print(f"✅ Ambiente pronto. Morph: {morph.__version__} | TestSuite: {testsuite.__version__}")


✅ Ambiente pronto. Morph: 1.1.2 | TestSuite: 1.1.2


#### Executando os Testes {.unnumbered}

Após implementar a solução, execute `TestSuite("EP06_01.extensão").run()` em uma nova célula, substituindo `extensão` pela linguagem utilizada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema obtém automaticamente os casos de teste do repositório do curso, executa o programa e apresenta o resultado da avaliação.

Em Python, também é possível testar a solução diretamente a partir de uma *string*, sem a necessidade de salvar o código em um arquivo. Para isso, armazene o programa em uma variável e utilize o método `run_code`:

```python
codigo = """
# ... seu código aqui ...
"""

TestSuite("EP06_01").run_code(codigo)
```

### EP06_01 🟢 Avaliação de Segmentação por IoU (*Intersection over Union*)

Ao longo deste capítulo, diversas etapas do *pipeline* produzem **máscaras binárias**, como na segmentação de documentos, na localização de *QRCodes* e na detecção de defeitos. Para avaliar objetivamente a qualidade dessas segmentações, é necessário compará-las com uma máscara de referência (*ground truth*).

Uma das métricas mais utilizadas para esse fim é a **IoU** (*Intersection over Union*, ou Interseção sobre União), definida como a razão entre a área de interseção e a área de união de duas máscaras binárias. Quanto maior o valor da IoU, maior a concordância entre a segmentação produzida pelo algoritmo e a referência.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (número de linhas) e $C$ (número de colunas).
2. **Máscara de referência:** Ler os $L \times C$ elementos binários (0 ou 1) da matriz `ref`.
3. **Máscara predita:** Ler os $L \times C$ elementos binários (0 ou 1) da matriz `pred`.
4. **Interseção:** Contar o número de posições $(i,j)$ para as quais `ref[i][j] = 1` e `pred[i][j] = 1`.
5. **União:** Contar o número de posições $(i,j)$ para as quais `ref[i][j] = 1` ou `pred[i][j] = 1`.
6. **Caso degenerado:** Se a união for igual a $0$, definir $\mathrm{IoU}=1{,}0$, pois ambas as máscaras são vazias.
7. **Cálculo:** Caso a união seja maior que zero, calcular

$$
\mathrm{IoU}=
\frac{|\mathrm{Intersecao}|}
{|\mathrm{Uniao}|}.
$$

8. **Classificação:** Determinar a classificação qualitativa utilizando o valor de IoU **antes** do arredondamento.
9. **Arredondamento:** Exibir a IoU com quatro casas decimais.
10. **Saída:** Imprimir, nessa ordem, a interseção, a união, a IoU e a classificação.

#### 📌 Restrições Computacionais

- Se a união for igual a $0$, não deve ser realizada a divisão; a IoU deve ser definida como $1{,}0$.
- As faixas de classificação utilizam comparações não estritas ($\geq$).
- A classificação deve ser realizada utilizando o valor da IoU em precisão completa, antes do arredondamento para exibição.

#### 🧠 Fundamentação Teórica

A IoU é definida por

$$
\mathrm{IoU}=
\frac{|R\cap P|}
{|R\cup P|},
$$

em que:

- $R$ representa o conjunto de pixels pertencentes à máscara de referência;
- $P$ representa o conjunto de pixels pertencentes à máscara predita;
- $|R\cap P|$ corresponde ao número de pixels pertencentes simultaneamente às duas máscaras;
- $|R\cup P|$ corresponde ao número de pixels pertencentes a pelo menos uma das máscaras.

| Faixa de IoU | Classificação | Interpretação |
|---|---|---|
| $\mathrm{IoU}\geq0{,}90$ | `EXCELENTE` | Concordância muito elevada entre as máscaras. |
| $0{,}70\leq\mathrm{IoU}<0{,}90$ | `BOM` | Pequenas diferenças entre as máscaras. |
| $0{,}50\leq\mathrm{IoU}<0{,}70$ | `ACEITAVEL` | Concordância parcial entre as máscaras. |
| $\mathrm{IoU}<0{,}50$ | `RUIM` | Baixa concordância entre as máscaras. |

A IoU depende apenas da sobreposição entre as máscaras e, portanto, é independente do tamanho da imagem.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

- Linha 1: inteiro $L$.
- Linha 2: inteiro $C$.
- Próximas $L$ linhas: elementos binários (0 ou 1) da matriz `ref`.
- Próximas $L$ linhas: elementos binários (0 ou 1) da matriz `pred`.

**Saída:**

- Linha 1: `Intersecao: X`
- Linha 2: `Uniao: Y`
- Linha 3: `IoU: Z`
- Linha 4: `Classificacao: NOME`

O valor de `IoU` deve ser impresso com quatro casas decimais.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 2<br>2<br>1 1<br>0 0<br>1 0<br>0 0 | Intersecao: 1<br>Uniao: 2<br>IoU: 0.5000<br>Classificacao: ACEITAVEL | A metade da região de referência foi corretamente segmentada. |
| 2<br>2<br>0 0<br>0 0<br>0 0<br>0 0 | Intersecao: 0<br>Uniao: 0<br>IoU: 1.0000<br>Classificacao: EXCELENTE | Ambas as máscaras são vazias; por convenção, $\mathrm{IoU}=1{,}0$. |

In [106]:
#| label: fig-06-sim-ep01
#| fig-cap: "Simulador: IoU entre máscara de referência e máscara predita"
#| echo: false
#| output: true

from IPython.display import HTML

HTML("""
<div id="sim-ep0601" style="background:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">

<div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
<span>🎮 Simulador: IoU (Intersection over Union)</span>
<span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">
IoU = |A∩B| / |A∪B|
</span>
</div>

<div style="padding:20px;background:white;">

<p style="font-size:11px;color:#777;text-align:center;">
Desloque e redimensione a máscara predita.
</p>

<div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">

<div style="display:flex;justify-content:space-between;">
<label style="font-size:12px;font-weight:bold;color:#2980b9;">Deslocamento horizontal</label>
<span id="ep0601_vdx" style="font-family:monospace;font-weight:bold;color:#2980b9;">0</span>
</div>

<input id="ep0601_dx"
type="range"
min="-3"
max="3"
step="1"
value="0"
style="width:100%;accent-color:#2980b9;">

<div style="display:flex;justify-content:space-between;margin-top:12px;">
<label style="font-size:12px;font-weight:bold;color:#8e44ad;">Deslocamento vertical</label>
<span id="ep0601_vdy" style="font-family:monospace;font-weight:bold;color:#8e44ad;">0</span>
</div>

<input id="ep0601_dy"
type="range"
min="-3"
max="3"
step="1"
value="0"
style="width:100%;accent-color:#8e44ad;">

<div style="display:flex;justify-content:space-between;margin-top:12px;">
<label style="font-size:12px;font-weight:bold;color:#16a34a;">Lado do quadrado</label>
<span id="ep0601_vsz" style="font-family:monospace;font-weight:bold;color:#16a34a;">6</span>
</div>

<input id="ep0601_sz"
type="range"
min="2"
max="8"
step="1"
value="6"
style="width:100%;accent-color:#16a34a;">

</div>

<div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:20px;">

<div style="text-align:center;">
<p style="font-size:11px;font-weight:bold;">Referência</p>
<div id="ep0601_ref"
style="display:grid;grid-template-columns:repeat(10,24px);gap:2px;justify-content:center;"></div>
</div>

<div style="text-align:center;">
<p style="font-size:11px;font-weight:bold;">Predita</p>
<div id="ep0601_pred"
style="display:grid;grid-template-columns:repeat(10,24px);gap:2px;justify-content:center;"></div>
</div>

<div style="text-align:center;">
<p style="font-size:11px;font-weight:bold;">Sobreposição</p>
<div id="ep0601_mix"
style="display:grid;grid-template-columns:repeat(10,24px);gap:2px;justify-content:center;"></div>
</div>

</div>

<div id="ep0601_dbg"
style="margin-top:20px;background:#e3f2fd;border-radius:8px;padding:12px;border:1px solid #bbdefb;font-family:monospace;font-size:12px;text-align:center;">
</div>

</div>
</div>

<script>
(function(){

function init(root){

if(!root) return;
if(root.dataset.ready==="1") return;
root.dataset.ready="1";

var dx=root.querySelector("#ep0601_dx");
var dy=root.querySelector("#ep0601_dy");
var sz=root.querySelector("#ep0601_sz");

var vdx=root.querySelector("#ep0601_vdx");
var vdy=root.querySelector("#ep0601_vdy");
var vsz=root.querySelector("#ep0601_vsz");

var gRef=root.querySelector("#ep0601_ref");
var gPred=root.querySelector("#ep0601_pred");
var gMix=root.querySelector("#ep0601_mix");

var dbg=root.querySelector("#ep0601_dbg");

var N=10;

var ref={x:2,y:2,w:6,h:6};

function inside(x,y,r){
return x>=r.x && x<r.x+r.w && y>=r.y && y<r.y+r.h;
}

function pixel(color){

var d=document.createElement("div");

d.style.width="24px";
d.style.height="24px";
d.style.border="1px solid #ddd";
d.style.boxSizing="border-box";
d.style.background=color;

return d;

}

function classe(i){

if(i>=0.90) return "Excelente";
if(i>=0.75) return "Muito boa";
if(i>=0.50) return "Aceitável";
return "Ruim";

}

function render(){

vdx.innerHTML=dx.value;
vdy.innerHTML=dy.value;
vsz.innerHTML=sz.value;

gRef.innerHTML="";
gPred.innerHTML="";
gMix.innerHTML="";

var pred={
x:ref.x+parseInt(dx.value),
y:ref.y+parseInt(dy.value),
w:parseInt(sz.value),
h:parseInt(sz.value)
};

var inter=0;
var uniao=0;

for(var y=0;y<N;y++){

for(var x=0;x<N;x++){

var r=inside(x,y,ref);
var p=inside(x,y,pred);

gRef.appendChild(pixel(r?"#7fdc92":"white"));
gPred.appendChild(pixel(p?"#7fbfff":"white"));

if(r&&p){
gMix.appendChild(pixel("#9b59b6"));
inter++;
}
else if(r){
gMix.appendChild(pixel("#7fdc92"));
uniao++;
}
else if(p){
gMix.appendChild(pixel("#7fbfff"));
uniao++;
}
else{
gMix.appendChild(pixel("white"));
}

if(r&&p) uniao++;

}

}

var iou=inter/uniao;

dbg.innerHTML=
"<b>Interseção</b> = "+inter+
" pixels &nbsp;&nbsp;&nbsp;"+
"<b>União</b> = "+uniao+
" pixels"+
"<br><br>"+
"IoU = <b>"+inter+" / "+uniao+
" = "+iou.toFixed(4)+"</b>"+
"<br><br>"+
"<span style='font-weight:bold;color:#1565c0;'>"+
classe(iou)+
"</span>";

}

dx.oninput=render;
dy.oninput=render;
sz.oninput=render;

render();

}

function tryInit(){

var all=document.querySelectorAll('[id="sim-ep0601"]');

if(all.length===0) return false;

init(all[all.length-1]);

return true;

}

var t=0;

var id=setInterval(function(){

t++;

if(tryInit() || t>20)
clearInterval(id);

},50);

})();
</script>
""")

In [107]:
%%writefile EP06_01.py
# Código Python

Overwriting EP06_01.py


In [108]:
TestSuite("EP06_01.py").run()

### EP06_02 🟢 Filtro de Marcadores por Circularidade

Após a segmentação de uma imagem, é comum que diversos componentes conexos sejam identificados. Em aplicações como a retificação de documentos, apenas alguns desses componentes correspondem aos marcadores de referência utilizados para o alinhamento da imagem. Um critério frequentemente empregado para selecionar esses marcadores é a **circularidade**, que mede o quão próxima a forma de um componente está de um círculo.

Neste exercício, cada componente é descrito por sua área $A$ e seu perímetro $P$. O objetivo é calcular sua circularidade e decidir, a partir de um limiar fornecido, se o componente deve ser aceito ou rejeitado como candidato a marcador.

#### 📋 Diretrizes de Implementação

1. **Quantidade:** Ler o inteiro $N$ (número de candidatos) e o limiar de circularidade $C_{\text{limiar}}$ (número real).
2. **Dados dos candidatos:** Para cada um dos $N$ candidatos, ler a área $A$ (inteiro) e o perímetro $P$ (número real).
3. **Circularidade:** Calcular $C=\frac{4\pi A}{P^2}$, em que:

- $A$ é a área do componente;
- $P$ é o perímetro do componente;
- $C$ é a circularidade.

4. **Caso degenerado:** Se $P=0$, considerar $C=0$ e classificar diretamente o candidato como `REJEITADO`.
5. **Classificação:** Se $C>C_{\text{limiar}}$, classificar o candidato como `ACEITO`; caso contrário, classificá-lo como `REJEITADO`.
6. **Arredondamento:** Exibir o valor de $C$ com quatro casas decimais.
7. **Saída:** Para cada candidato, imprimir o valor de $C$ seguido da classificação. Ao final, imprimir o número total de candidatos aceitos.

#### 📌 Restrições Computacionais

- Utilizar a constante $\pi$ da biblioteca padrão da linguagem (por exemplo, `math.pi`), sem aproximações.
- A comparação deve ser realizada com o valor de $C$ em precisão completa, antes do arredondamento para exibição.
- O critério de aceitação é estrito ($C>C_{\text{limiar}}$).
- Se $P=0$, a divisão não deve ser realizada.

#### 🧠 Fundamentação Teórica

A circularidade é um descritor geométrico definido por $C=\frac{4\pi A}{P^2}$, em que:

- $A$ é a área do componente;
- $P$ é o perímetro do componente;
- $C$ é a circularidade.

Para um círculo perfeito, $C=1$. À medida que a forma se torna mais alongada ou irregular, o perímetro cresce mais rapidamente que a área, reduzindo o valor de $C$.

| Forma | Circularidade aproximada | Interpretação |
|---|---:|---|
| Círculo | $1{,}0000$ | Forma circular. |
| Quadrado | $0{,}7854$ | Forma aproximadamente compacta. |
| Forma alongada ou irregular | $C\ll1$ | Baixa circularidade. |
| $P=0$ | $0$ (convenção adotada) | Contorno degenerado. |

A circularidade é invariante à translação, à rotação e à escala, sendo amplamente utilizada para distinguir componentes aproximadamente circulares de outros formatos.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

- Linha 1: inteiro $N$.
- Linha 2: número real $C_{\text{limiar}}$.
- Próximas $N$ linhas: área $A$ (inteiro) e perímetro $P$ (real), separados por espaço.

**Saída:**

- Uma linha para cada candidato, no formato `C ACEITO` ou `C REJEITADO`, com $C$ apresentado com quatro casas decimais.
- Última linha: `Total aceitos: X`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3<br>0.6<br>78 31.4<br>100 40<br>50 60 | 0.9941 ACEITO<br>0.7854 ACEITO<br>0.1745 REJEITADO<br>Total aceitos: 2 | Candidato aproximadamente circular, forma compacta e forma alongada. |
| 1<br>0.9<br>10 0 | 0.0000 REJEITADO<br>Total aceitos: 0 | Perímetro nulo: contorno degenerado. |

In [109]:
#| label: fig-06-sim-ep02
#| fig-cap: "Simulador: Filtro de Marcadores por Circularidade"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0602" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Filtro de Marcadores por Circularidade</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 C = 4πA / P²</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste o limiar e observe quais candidatos (discos, quadrados e formas irregulares) sobrevivem ao filtro.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Limiar de circularidade (C_limiar)</label>
        <span id="ep0602_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">0.60</span>
      </div>
      <input id="ep0602_sl" style="width:100%;accent-color:#2980b9;" max="0.99" min="0.05" step="0.01" type="range" value="0.60">
    </div>
    <div id="ep0602_cards" style="display:grid;grid-template-columns:repeat(5,1fr);gap:10px;"></div>
    <div id="ep0602_debug" style="margin-top:20px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var candidatos = [
      {nome:"Disco", A:78, P:31.4},
      {nome:"Quadrado", A:100, P:40},
      {nome:"Retângulo", A:60, P:44},
      {nome:"Rasura", A:50, P:60},
      {nome:"Ponto", A:10, P:0}
    ];
    var slEl = root.querySelector('#ep0602_sl');
    var vlEl = root.querySelector('#ep0602_vl');
    var cards = root.querySelector('#ep0602_cards');
    var dbg = root.querySelector('#ep0602_debug');

    function render(){
      var th = parseFloat(slEl.value);
      vlEl.textContent = th.toFixed(2);
      cards.innerHTML = '';
      var aceitos = 0;
      candidatos.forEach(function(c){
        var C = (c.P === 0) ? 0 : (4*Math.PI*c.A)/(c.P*c.P);
        var ok = c.P !== 0 && C > th;
        if(ok) aceitos++;
        var div = document.createElement('div');
        div.style.cssText = 'text-align:center;border-radius:10px;padding:10px 6px;font-size:11px;' +
          (ok ? 'background:#dcfce7;border:1px solid #86efac;color:#166534;' : 'background:#fee2e2;border:1px solid #fca5a5;color:#991b1b;');
        div.innerHTML = '<div style="font-weight:700;">'+c.nome+'</div>' +
          '<div style="font-family:monospace;margin:4px 0;">A='+c.A+'<br>P='+c.P+'</div>' +
          '<div style="font-family:monospace;font-weight:700;">C='+C.toFixed(4)+'</div>' +
          '<div style="font-weight:700;">'+(ok?'ACEITO':'REJEITADO')+'</div>';
        cards.appendChild(div);
      });
      dbg.textContent = 'C_limiar=' + th.toFixed(2) + '  |  Candidatos aceitos: ' + aceitos + '/' + candidatos.length;
    }
    slEl.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0602');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


In [110]:
%%writefile EP06_02.py
# Código Python

Overwriting EP06_02.py


In [111]:
TestSuite("EP06_02.py").run()

### EP06_03 🟡 Classificação de Marcações em Folhas de Resposta (OMR)

Após a retificação da folha e a segmentação dos quadros de respostas, o MCTest estima, para cada bolha, um **grau de preenchimento**, representado por um valor entre $0$ e $100$. A partir desses valores, o sistema deve determinar automaticamente a alternativa marcada, identificando também questões em branco e casos de múltiplas marcações.

Neste exercício, você implementará essa etapa de decisão do *pipeline* de OMR. A classificação depende de um limiar de preenchimento: pequenas variações nesse valor podem alterar o resultado da leitura automática.

#### 📋 Diretrizes de Implementação

1. **Parâmetros:** Ler os inteiros $Q$ (número de questões) e $K$ (número de alternativas por questão, com $2 \le K \le 26$) e o limiar de preenchimento $\mathrm{Th}$ (número real entre $0$ e $100$).
2. **Graus de preenchimento:** Para cada uma das $Q$ questões, ler os $K$ valores reais correspondentes às alternativas `A`, `B`, `C`, ..., na ordem de entrada.
3. **Contagem de marcações:** Para cada questão, contar quantas alternativas possuem grau de preenchimento **estritamente maior** que $\mathrm{Th}$.
4. **Classificação:**
   - Se nenhuma alternativa exceder $\mathrm{Th}$, classificar a questão como `BRANCO`.
   - Se exatamente uma alternativa exceder $\mathrm{Th}$, imprimir a letra correspondente (`A`, `B`, `C`, ...).
   - Se duas ou mais alternativas excederem $\mathrm{Th}$, classificar a questão como `DUPLA_MARCACAO`.
5. **Saída por questão:** Imprimir, na ordem de leitura, a classificação de cada questão.
6. **Totais:** Ao final, imprimir o número de questões `OK` (uma única marcação), `BRANCO` e `DUPLA_MARCACAO`.

#### 📌 Restrições Computacionais

* **Comparação estrita:** apenas valores maiores que $\mathrm{Th}$ são considerados marcações válidas; valores exatamente iguais ao limiar não devem ser contabilizados.
* **Letras das alternativas:** o índice $0$ corresponde à alternativa `A`, o índice $1$ à alternativa `B` e assim sucessivamente.
* **Múltiplas marcações:** sempre que duas ou mais alternativas excederem o limiar, a classificação deve ser `DUPLA_MARCACAO`, independentemente dos respectivos graus de preenchimento.

#### 🧠 Fundamentação Teórica

| Situação | Classificação | Interpretação |
|---|---|---|
| Exatamente uma alternativa acima do limiar | Letra da alternativa | Resposta válida |
| Nenhuma alternativa acima do limiar | `BRANCO` | Questão não respondida |
| Duas ou mais alternativas acima do limiar | `DUPLA_MARCACAO` | Resposta ambígua |

O limiar de preenchimento controla a sensibilidade do algoritmo. Valores muito baixos tendem a aumentar o número de `DUPLA_MARCACAO`, enquanto valores muito altos podem aumentar a quantidade de questões classificadas como `BRANCO`.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $Q$.
* Linha 2: Inteiro $K$.
* Linha 3: Número real $\mathrm{Th}$.
* Próximas $Q$ linhas: $K$ números reais, correspondentes aos graus de preenchimento das alternativas.
* Linha 1: Inteiro $Q$ e $K$.

**Saída:**

* $Q$ linhas, cada uma contendo a classificação da respectiva questão.
* Linha final: `OK: x  BRANCO: y  DUPLA_MARCACAO: z`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3<br>4<br>50<br>10 85 5 12<br>20 15 18 22<br>90 88 10 5 | B<br>BRANCO<br>DUPLA_MARCACAO<br>OK: 1  BRANCO: 1  DUPLA_MARCACAO: 1 | Na primeira questão apenas `B` supera o limiar; na segunda nenhuma alternativa o supera; na terceira, `A` e `B` excedem o limiar. |
| 1<br>2<br>50.0<br>50 50 | BRANCO<br>OK: 0  BRANCO: 1  DUPLA_MARCACAO: 0 | Valores iguais ao limiar não são considerados marcações válidas. |

In [112]:
#| label: fig-06-sim-ep03
#| fig-cap: "Simulador: Classificação de Marcações OMR"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0603" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Classificação de Marcações OMR</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 4 alternativas</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste o grau de preenchimento de cada bolha (A–D) e o limiar, e observe a classificação da questão.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:4px;">
        <label style="font-size:12px;font-weight:bold;color:#a16207;">Limiar de preenchimento (Th)</label>
        <span id="ep0603_vth" style="font-family:monospace;font-weight:bold;color:#a16207;">50</span>
      </div>
      <input id="ep0603_th" style="width:100%;accent-color:#a16207;" max="100" min="0" step="1" type="range" value="50">
    </div>
    <div id="ep0603_bubbles" style="display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin-bottom:16px;"></div>
    <div id="ep0603_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:12px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var letras = ['A','B','C','D'];
    var valores = [10, 85, 5, 12];
    var thEl = root.querySelector('#ep0603_th'), vthEl = root.querySelector('#ep0603_vth');
    var box = root.querySelector('#ep0603_bubbles');
    var dbg = root.querySelector('#ep0603_debug');

    box.innerHTML = '';
    var sliders = [];
    letras.forEach(function(L, i){
      var col = document.createElement('div');
      col.style.cssText = 'text-align:center;';
      col.innerHTML = '<div style="font-weight:700;font-size:13px;margin-bottom:4px;">'+L+'</div>' +
        '<input type="range" min="0" max="100" step="1" value="'+valores[i]+'" style="width:100%;accent-color:#7c3aed;" id="ep0603_b'+i+'">' +
        '<div id="ep0603_v'+i+'" style="font-family:monospace;font-size:11px;margin-top:4px;">'+valores[i]+'%</div>';
      box.appendChild(col);
      sliders.push(col.querySelector('#ep0603_b'+i));
    });

    function render(){
      var th = parseFloat(thEl.value);
      vthEl.textContent = th.toFixed(0);
      var marcadas = [];
      sliders.forEach(function(s, i){
        var v = parseFloat(s.value);
        root.querySelector('#ep0603_v'+i).textContent = v.toFixed(0) + '%';
        if(v > th) marcadas.push(letras[i]);
      });
      var resultado;
      if(marcadas.length === 0) resultado = 'BRANCO';
      else if(marcadas.length === 1) resultado = marcadas[0];
      else resultado = 'DUPLA_MARCACAO (' + marcadas.join(',') + ')';
      dbg.textContent = 'Classificacao da questao: ' + resultado;
    }
    sliders.forEach(function(s){ s.addEventListener('input', render); });
    thEl.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0603');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


In [113]:
%%writefile EP06_03.py
# Código Python

Overwriting EP06_03.py


In [114]:
TestSuite("EP06_03.py").run()

### EP06_04 🟡 Estimador de Inclinação por Mediana Angular (*Deskew*)

Após a detecção de bordas e a aplicação da Transformada de Hough, obtém-se um conjunto de retas candidatas à orientação predominante do documento. Cada reta fornece uma estimativa do ângulo de inclinação, calculada por

$$
\text{ângulo} = \operatorname{rad2deg}(\theta) - 90.
$$

Entretanto, nem todas as retas correspondem às linhas do documento: algumas resultam de ruídos, sombras ou outros elementos da imagem. Neste exercício, você implementará a etapa de estimação robusta do ângulo de inclinação, filtrando os valores plausíveis e calculando sua mediana.

#### 📋 Diretrizes de Implementação

1. **Quantidade:** Ler o inteiro $M$, correspondente ao número de ângulos estimados.
2. **Ângulos:** Ler os $M$ valores reais, em graus.
3. **Filtragem:** Manter apenas os ângulos que satisfaçam **estritamente** $-45 < \text{ângulo} < 45$.
4. **Ausência de candidatos:** Se nenhum ângulo permanecer após a filtragem, imprimir exatamente `SEM_CORRECAO`.
5. **Mediana:** Caso existam ângulos válidos:
   - se a quantidade for ímpar, a mediana é o elemento central da sequência ordenada;
   - se for par, a mediana é a média aritmética dos dois elementos centrais.
6. **Saída:** Imprimir a mediana arredondada para duas casas decimais (arredondamento padrão, *round half away from zero*, , com `np.floor(img + 0.5)`).

#### 📌 Restrições Computacionais

* **Intervalo aberto:** ângulos iguais a $-45$ ou $45$ não devem ser considerados.
* **Precisão:** calcular a mediana utilizando os valores originais; o arredondamento deve ser realizado apenas na saída.
* **Caso vazio:** se não houver ângulos válidos, nenhuma mediana deve ser calculada.

#### 🧠 Fundamentação Teórica

| Situação | Resultado |
|---|---|
| Maioria dos ângulos concentrada em torno da inclinação real | A mediana aproxima a orientação do documento. |
| Poucos ângulos discrepantes (*outliers*) | A mediana sofre pouca influência desses valores. |
| Ângulos fora do intervalo $(-45^\circ,45^\circ)$ | São descartados antes do cálculo. |
| Nenhum ângulo válido | Não é aplicada correção (`SEM_CORRECAO`). |

A mediana é utilizada por ser mais robusta que a média na presença de poucos valores discrepantes, produzindo uma estimativa mais estável da inclinação predominante do documento.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $M$.
* Linha 2: $M$ números reais, correspondentes aos ângulos em graus.

**Saída:**

* Uma única linha contendo o ângulo estimado, com duas casas decimais, ou a palavra `SEM_CORRECAO` caso nenhum ângulo seja válido.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 5<br>-50 -10.5 2.3 2.3 47 | 2.30 | Apenas os ângulos no intervalo $(-45,45)$ são considerados; a mediana é $2{,}3$. |
| 4<br>-46 50 45 -45 | SEM_CORRECAO | Nenhum ângulo pertence ao intervalo aberto $(-45,45)$. |

In [115]:
#| label: fig-06-sim-ep04
#| fig-cap: "Simulador: Estimador de Inclinação por Mediana Angular"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0604" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Mediana Angular (Deskew)</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 mediana(-45° &lt; θ &lt; 45°)</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Arraste os pontos de ruído para fora ou para dentro do intervalo válido e veja como a mediana permanece estável.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#7c3aed;">Ângulo do ruído extra (grau)</label>
        <span id="ep0604_vl" style="font-family:monospace;font-weight:bold;color:#7c3aed;">47</span>
      </div>
      <input id="ep0604_sl" style="width:100%;accent-color:#7c3aed;" max="80" min="-80" step="1" type="range" value="47">
    </div>
    <div id="ep0604_pts" style="display:flex;gap:8px;flex-wrap:wrap;justify-content:center;margin-bottom:16px;"></div>
    <div id="ep0604_debug" style="background:#ede9fe;border-radius:8px;padding:10px;border:1px solid #c4b5fd;font-family:monospace;font-size:11px;color:#5b21b6;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var base = [-10.5, 2.3, 2.3];
    var slEl = root.querySelector('#ep0604_sl');
    var vlEl = root.querySelector('#ep0604_vl');
    var ptsEl = root.querySelector('#ep0604_pts');
    var dbg = root.querySelector('#ep0604_debug');

    function median(arr){
      var a = arr.slice().sort(function(x,y){return x-y;});
      var n = a.length;
      if(n===0) return null;
      var mid = Math.floor(n/2);
      return (n%2===1) ? a[mid] : (a[mid-1]+a[mid])/2;
    }

    function render(){
      var extra = parseFloat(slEl.value);
      vlEl.textContent = extra;
      var todos = base.concat([extra, -50]);
      var validos = todos.filter(function(a){ return a > -45 && a < 45; });
      ptsEl.innerHTML = '';
      todos.forEach(function(a){
        var ok = a > -45 && a < 45;
        var div = document.createElement('div');
        div.style.cssText = 'padding:8px 12px;border-radius:8px;font-family:monospace;font-size:12px;font-weight:700;' +
          (ok ? 'background:#dcfce7;border:1px solid #86efac;color:#166534;' : 'background:#fee2e2;border:1px solid #fca5a5;color:#991b1b;');
        div.textContent = a + '°';
        ptsEl.appendChild(div);
      });
      var med = median(validos);
      dbg.textContent = 'Válidos: [' + validos.join(', ') + ']  |  Mediana estimada: ' +
        (med===null ? 'SEM_CORRECAO' : med.toFixed(2)+'°');
    }
    slEl.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0604');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


In [116]:
%%writefile EP06_04.py
# Código Python

Overwriting EP06_04.py


In [117]:
TestSuite("EP06_04.py").run()

### EP06_05 🟠 Normalização de Fundo por Divisão (Correção de Iluminação)

Um formulário foi fotografado sob iluminação não uniforme, fazendo com que um lado da folha apareça mais claro que o outro. Nessas condições, a limiarização global por Otsu pode produzir resultados insatisfatórios, pois um único limiar não separa adequadamente texto e fundo em toda a imagem. A solução apresentada no capítulo consiste em **normalizar o fundo**, dividindo a imagem original por uma versão fortemente suavizada de si mesma, que representa a iluminação de baixa frequência.

Neste exercício, a imagem original e o fundo suavizado (equivalente ao resultado de um `cv2.GaussianBlur` com $\sigma$ elevado) já são fornecidos. Sua tarefa é implementar a etapa de normalização que produz a imagem corrigida.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Imagem original:** Ler os $L \times C$ valores inteiros da matriz `img` (intensidades entre 0 e 255).
3. **Fundo estimado:** Ler os $L \times C$ valores inteiros da matriz `bg` (intensidades entre 0 e 255, sempre estritamente maiores que zero).
4. **Normalização:** Para cada posição $(i,j)$, calcular
$$
\text{valor}(i,j)=
\frac{\text{img}(i,j)}{\text{bg}(i,j)}\times255.
$$
5. **Arredondamento:** Arredondar o resultado para o inteiro mais próximo (*round half away from zero*, com `np.floor(img + 0.5)`).
6. **Saturação:** Limitar o valor obtido ao intervalo $[0,255]$.
7. **Saída:** Imprimir a matriz `img_norm` resultante.

#### 📌 Restrições Computacionais

* **Divisão por zero:** a entrada garante $\text{bg}(i,j)>0$ em todas as posições.
* **Ordem das operações:** primeiro arredondar, depois aplicar a saturação.
* **Processamento independente:** cada pixel deve ser normalizado individualmente, sem utilizar informações dos pixels vizinhos.

#### 🧠 Fundamentação Teórica

| Situação | Efeito da normalização |
|---|---|
| $\text{img}(i,j)=\text{bg}(i,j)$ | Resultado igual a $255$, correspondente ao fundo normalizado. |
| $\text{img}(i,j)<\text{bg}(i,j)$ | Resultado menor que $255$, preservando regiões mais escuras, como texto. |
| $\text{img}(i,j)>\text{bg}(i,j)$ | Resultado superior a $255$, posteriormente saturado. |
| Fundo com iluminação não uniforme | A divisão reduz as variações lentas de iluminação, tornando a imagem mais homogênea. |

A divisão pelo fundo estimado reduz os efeitos da iluminação não uniforme e preserva o contraste entre o primeiro plano e o fundo, facilitando as etapas posteriores de segmentação.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Próximas $L$ linhas: elementos da matriz `img`.
* Próximas $L$ linhas: elementos da matriz `bg`.

**Saída:**

* Matriz `img_norm`, com $L$ linhas e $C$ colunas, contendo valores inteiros separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 2<br>2<br>60 120<br>180 40<br>100 100<br>200 80 | 153 255<br>230 128 | Valores superiores a $255$ devem ser saturados; $180/200\times255=229{,}5$ resulta em $230$ após o arredondamento. |
| 1<br>3<br>30 60 90<br>60 60 60 | 128 255 255 | Apenas o primeiro valor permanece abaixo de $255$ após a normalização. |

In [118]:
#| label: fig-06-sim-ep05
#| fig-cap: "Simulador: Normalização de Fundo por Divisão"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0605" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Normalização de Fundo por Divisão</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟠 img/bg × 255</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste o gradiente de fundo (esquerda x direita) e observe como a normalização cancela a variação de iluminação.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#c2410c;">Intensidade do fundo à esquerda (bg_esq)</label>
        <span id="ep0605_vl" style="font-family:monospace;font-weight:bold;color:#c2410c;">100</span>
      </div>
      <input id="ep0605_sl" style="width:100%;accent-color:#c2410c;" max="220" min="40" step="5" type="range" value="100">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:14px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;">img (original)</p>
        <div id="ep0605_g_img" style="display:grid;grid-template-columns:repeat(4,40px);gap:3px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;">bg (fundo suavizado)</p>
        <div id="ep0605_g_bg" style="display:grid;grid-template-columns:repeat(4,40px);gap:3px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;">img_norm (saída)</p>
        <div id="ep0605_g_out" style="display:grid;grid-template-columns:repeat(4,40px);gap:3px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0605_debug" style="margin-top:16px;background:#ffedd5;border-radius:8px;padding:10px;border:1px solid #fdba74;font-family:monospace;font-size:11px;color:#9a3412;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var linha_img = [90, 90, 90, 90];
    var slEl = root.querySelector('#ep0605_sl');
    var vlEl = root.querySelector('#ep0605_vl');
    var gImg = root.querySelector('#ep0605_g_img');
    var gBg  = root.querySelector('#ep0605_g_bg');
    var gOut = root.querySelector('#ep0605_g_out');
    var dbg  = root.querySelector('#ep0605_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }
    function cellStyle(v){
      var g = Math.max(0, Math.min(255, v));
      return 'width:40px;height:40px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:9px;font-weight:bold;font-family:monospace;border:1px solid #ccc;' +
             'background:rgb('+g+','+g+','+g+');color:'+(g>140?'#000':'#fff')+';';
    }

    function render(){
      var bgEsq = parseInt(slEl.value);
      vlEl.textContent = bgEsq;
      // gradiente linear de bgEsq até 200 na direita, 4 colunas
      var bg = [];
      for(var j=0;j<4;j++){
        bg.push(Math.round(bgEsq + (200-bgEsq)*j/3));
      }
      gImg.innerHTML=''; gBg.innerHTML=''; gOut.innerHTML='';
      var out = [];
      for(var j=0;j<4;j++){
        var v = linha_img[j]/bg[j]*255;
        var r = roundHalfAway(v);
        var sat = Math.max(0, Math.min(255, r));
        out.push(sat);
        var ci = document.createElement('div'); ci.style.cssText = cellStyle(linha_img[j]); ci.textContent = linha_img[j];
        gImg.appendChild(ci);
        var cb = document.createElement('div'); cb.style.cssText = cellStyle(bg[j]); cb.textContent = bg[j];
        gBg.appendChild(cb);
        var co = document.createElement('div'); co.style.cssText = cellStyle(sat); co.textContent = sat;
        gOut.appendChild(co);
      }
      dbg.textContent = 'bg = [' + bg.join(', ') + ']  |  img_norm = [' + out.join(', ') + ']';
    }
    slEl.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0605');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


In [119]:
%%writefile EP06_05.py
# Código Python

Overwriting EP06_05.py


In [120]:
TestSuite("EP06_05.py").run()

### EP06_06 🔴 Mapa de Variância Local para Detecção de Textura

Uma fábrica de tecidos precisa inspecionar rolos de pano em tempo real, sem dispor de uma imagem de referência — cada rolo apresenta pequenas variações naturais. Nessa situação, a estratégia apresentada no capítulo consiste em analisar a **homogeneidade local da textura**: regiões uniformes apresentam baixa variância de intensidade em pequenas vizinhanças, enquanto riscos, manchas e falhas de fabricação produzem aumentos locais dessa variância.

Neste exercício, você implementará o núcleo desse método, calculando a variância local em uma janela deslizante e gerando uma máscara binária que identifica as regiões cuja variância excede um limiar.

#### 📋 Diretrizes de Implementação

1. **Dimensões e parâmetros:** Ler os inteiros $L$, $C$, $k$ (tamanho da janela, sempre ímpar) e $T$ (limiar de variância).
2. **Imagem:** Ler os $L \times C$ valores inteiros da matriz de textura (intensidades entre 0 e 255).
3. **Tratamento das bordas:** Quando a janela ultrapassar os limites da imagem, utilizar **replicação de borda**, isto é, repetir o valor do pixel válido mais próximo.
4. **Média local:** Para cada posição $(i,j)$, calcular
$$
\mu(i,j)=
\frac{1}{k^2}
\sum_{(p,q)\in\text{janela}}
\text{textura}(p,q).
$$
5. **Variância local:** Calcular a variância populacional da janela,
$$
\sigma^2(i,j)=
\frac{1}{k^2}
\sum_{(p,q)\in\text{janela}}
\left(\text{textura}(p,q)-\mu(i,j)\right)^2,
$$
ou, de forma equivalente,
$$
\sigma^2(i,j)=\overline{x^2}-\mu(i,j)^2,
$$
em que $\overline{x^2}$ representa a média dos quadrados das intensidades.

6. **Arredondamento:** Arredondar a variância para o inteiro mais próximo (*round half away from zero*, com `np.floor(res_norm + 0.5)`).

7. **Limiarização:** Definir $\text{máscara}(i,j)=1$ se a variância arredondada for **estritamente maior** que $T$; caso contrário, definir $\text{máscara}(i,j)=0$.

8. **Saída:** Imprimir a máscara binária resultante.

#### 📌 Restrições Computacionais

* **Replicação de borda:** utilizar o valor do pixel válido mais próximo sempre que a janela ultrapassar os limites da imagem.
* **Variância populacional:** utilizar denominador $k^2$, nunca $k^2-1$.
* **Comparação estrita:** a máscara deve ser calculada utilizando a condição $\sigma^2_{\text{arred}}>T$.
* **Janela ímpar:** o valor de $k$ é sempre ímpar, garantindo um pixel central.

#### 🧠 Fundamentação Teórica

| Situação | Variância local | Interpretação |
|---|---|---|
| Região uniforme | Baixa | Intensidades semelhantes na vizinhança. |
| Região contendo defeito | Alta | A presença de intensidades distintas aumenta a dispersão dos valores. |
| Janela pequena | Maior sensibilidade a detalhes e ruído | Detecta alterações localizadas. |
| Janela grande | Resposta mais suave | Evidencia defeitos maiores, porém reduz a precisão de sua localização. |

A variância local mede a dispersão das intensidades em uma vizinhança. Regiões homogêneas apresentam baixa variância, enquanto alterações na textura aumentam essa medida, permitindo identificar possíveis defeitos por meio de uma simples limiarização.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $k$ (ímpar).
* Linha 4: Inteiro $T$.
* Próximas $L$ linhas: elementos inteiros da matriz de textura.

**Saída:**

* Máscara binária (valores 0 ou 1), com $L$ linhas e $C$ colunas.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3<br>3<br>3<br>50<br>10 10 10<br>10 10 10<br>10 90 10 | 0 0 0<br>1 1 1<br>1 1 1 | O defeito aumenta a variância em todas as janelas que o contêm. |
| 2<br>2<br>3<br>5<br>100 100<br>100 100 | 0 0<br>0 0 | A textura é uniforme; a variância é nula em toda a imagem. |

In [121]:
#| label: fig-06-sim-ep06
#| fig-cap: "Simulador: Mapa de Variância Local para Detecção de Textura"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0606" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Variância Local (Detecção de Textura)</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🔴 σ² = média(x²) − média(x)²</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste o valor do "defeito" central e o limiar T; observe como a janela 3×3 espalha a detecção pela vizinhança (borda replicada).</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:4px;">
        <label style="font-size:12px;font-weight:bold;color:#dc2626;">Intensidade do defeito (posição central)</label>
        <span id="ep0606_vl_def" style="font-family:monospace;font-weight:bold;color:#dc2626;">90</span>
      </div>
      <input id="ep0606_sl_def" style="width:100%;accent-color:#dc2626;" max="255" min="10" step="5" type="range" value="90">
      <div style="display:flex;justify-content:space-between;margin:10px 0 4px;">
        <label style="font-size:12px;font-weight:bold;color:#dc2626;">Limiar T</label>
        <span id="ep0606_vl_t" style="font-family:monospace;font-weight:bold;color:#dc2626;">50</span>
      </div>
      <input id="ep0606_sl_t" style="width:100%;accent-color:#dc2626;" max="2000" min="0" step="10" type="range" value="50">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Textura (3×3)</p>
        <div id="ep0606_g_tex" style="display:grid;grid-template-columns:repeat(3,44px);gap:4px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Máscara de Defeito</p>
        <div id="ep0606_g_mask" style="display:grid;grid-template-columns:repeat(3,44px);gap:4px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0606_debug" style="margin-top:20px;background:#fee2e2;border-radius:8px;padding:10px;border:1px solid #fca5a5;font-family:monospace;font-size:11px;color:#991b1b;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var slDef = root.querySelector('#ep0606_sl_def');
    var vlDef = root.querySelector('#ep0606_vl_def');
    var slT   = root.querySelector('#ep0606_sl_t');
    var vlT   = root.querySelector('#ep0606_vl_t');
    var gTex  = root.querySelector('#ep0606_g_tex');
    var gMask = root.querySelector('#ep0606_g_mask');
    var dbg   = root.querySelector('#ep0606_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }
    function clampIdx(v, n){ return Math.max(0, Math.min(n-1, v)); }

    function render(){
      var defeito = parseInt(slDef.value);
      var T = parseInt(slT.value);
      vlDef.textContent = defeito;
      vlT.textContent = T;
      var N = 3;
      var tex = [[10,10,10],[10,10,10],[10,defeito,10]];
      gTex.innerHTML=''; gMask.innerHTML='';
      var mask = [];
      for(var i=0;i<N;i++){
        var row=[];
        for(var j=0;j<N;j++){
          var vals=[];
          for(var di=-1; di<=1; di++){
            for(var dj=-1; dj<=1; dj++){
              var pi = clampIdx(i+di, N), pj = clampIdx(j+dj, N);
              vals.push(tex[pi][pj]);
            }
          }
          var mean = vals.reduce(function(a,b){return a+b;},0)/vals.length;
          var meanSq = vals.reduce(function(a,b){return a+b*b;},0)/vals.length;
          var varr = meanSq - mean*mean;
          var varRound = roundHalfAway(varr);
          row.push(varRound > T ? 1 : 0);
        }
        mask.push(row);
      }
      for(var i=0;i<N;i++){
        for(var j=0;j<N;j++){
          var g = tex[i][j];
          var ct = document.createElement('div');
          ct.style.cssText = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:11px;font-weight:bold;font-family:monospace;border:1px solid #ccc;background:rgb('+g+','+g+','+g+');color:'+(g>140?'#000':'#fff')+';';
          ct.textContent = g;
          gTex.appendChild(ct);
          var m = mask[i][j];
          var cm = document.createElement('div');
          cm.style.cssText = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:12px;font-weight:bold;font-family:monospace;' +
            (m ? 'background:#fecaca;color:#7f1d1d;border:1px solid #f87171;' : 'background:#f3f4f6;color:#9ca3af;border:1px solid #e5e7eb;');
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }
      var total = mask.flat().reduce(function(a,b){return a+b;},0);
      dbg.textContent = 'defeito=' + defeito + '  |  T=' + T + '  |  Pixels marcados: ' + total + '/9';
    }
    slDef.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0606');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


In [122]:
%%writefile EP06_06.py
# Código Python

Overwriting EP06_06.py


In [123]:
TestSuite("EP06_06.py").run()

### EP06_07 🟣 *Pipeline* de Inspeção Industrial: Registro por Translação e Subtração

Em uma linha de produção, uma câmera fixa fotografa cada peça que passa pela esteira, comparando-a a uma imagem de referência sem defeitos. O problema: pequenas vibrações da esteira deslocam a peça em relação à posição de referência a cada captura. Se a subtração de imagens for aplicada diretamente, sem correção, o deslocamento por si só já gera diferenças enormes — **falsos positivos** que mascaram os defeitos reais.

Este é o exercício mais completo do capítulo: você deve **primeiro registrar** (alinhar geometricamente) a imagem capturada usando um deslocamento conhecido $(dx, dy)$, fornecido por um sensor de posição da esteira, e **só então aplicar a subtração** com limiarização, exatamente como descrito na seção de inspeção industrial.

#### 📋 Diretrizes de Implementação

1. **Dimensões e parâmetros:** Ler $L$, $C$ (dimensões das imagens), o deslocamento inteiro conhecido $dx, dy$ (podendo ser negativos) e o limiar de detecção $T$ (inteiro).
2. **Imagens:** Ler a matriz de referência (`ref`, $L\times C$, sem defeitos) e a matriz capturada (`cap`, $L\times C$, possivelmente deslocada e com defeito).
3. **Registro por translação:** Construir a imagem alinhada `alin` aplicando o deslocamento $(dx,dy)$ recebido:
$$
\text{alin}(i,j) = \begin{cases} \text{cap}(i+dy,\; j+dx), & \text{se } (i+dy,\ j+dx) \in [0,L)\times[0,C) \\ 0, & \text{caso contrário} \end{cases}
$$
4. **Preenchimento de borda:** As posições que "saem" da imagem capturada após o deslocamento recebem o valor **0** (*zero-padding* — fora do campo de visão da câmera; **note que este exercício usa zero, diferente da replicação de borda do EP06_06**).
5. **Diferença absoluta:** Calcular, pixel a pixel,
$$
\text{diff}(i,j) = |\text{ref}(i,j) - \text{alin}(i,j)|
$$
6. **Limiarização:** Definir $\text{máscara}(i,j) = 1$ se $\text{diff}(i,j) > T$; caso contrário, $\text{máscara}(i,j) = 0$.
7. **Saída:** Nesta ordem — (a) a matriz `alin` ($L\times C$); (b) a máscara de defeito ($L\times C$); (c) uma última linha com o total de pixels classificados como defeituosos.

#### 📌 Restrições Computacionais

* ***Zero-padding*, não replicação:** posições fora dos limites da imagem capturada, após o deslocamento, valem exatamente 0 — este é o ponto que mais diferencia este exercício do EP06_06.
* **Comparação estrita:** $\text{diff}(i,j) > T$.
* **Sinal de $(dx,dy)$:** o deslocamento pode ser positivo ou negativo; a fórmula do passo 3 deve ser aplicada literalmente, sem inverter os sinais.
* **Todos os valores são inteiros:** não há arredondamento nesta etapa.

#### 🧠 Fundamentação Teórica

| Etapa omitida | Consequência |
|---|---|
| Pular o registro geométrico | A borda inteira da imagem (introduzida pelo deslocamento) é marcada como "defeito" — falso positivo sistemático |
| Registro com $(dx,dy)$ incorreto | Peça e referência ficam desalinhadas; a subtração detecta contornos deslocados, não defeitos reais |
| Limiar $T$ muito baixo | Ruído de captura (variações de 1–2 níveis de cinza) é confundido com defeito |
| Limiar $T$ muito alto | Defeitos sutis deixam de ser detectados |

O registro geométrico e a subtração são etapas complementares: o primeiro garante que ambas as imagens representem exatamente a mesma cena no mesmo referencial espacial; o segundo isola o que realmente mudou entre elas — idealmente, apenas os defeitos.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Dois inteiros $dx$ e $dy$, separados por espaço.
* Linha 4: Inteiro $T$.
* Próximas $L$ linhas: elementos inteiros da matriz `ref`.
* Próximas $L$ linhas: elementos inteiros da matriz `cap`.

**Saída:**

* $L$ linhas com a matriz `alin`.
* $L$ linhas com a máscara de defeito (0/1).
* Última linha: `Total de pixels defeituosos: X`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3<br>3<br>1 0<br>30<br>50 50 50<br>50 50 50<br>50 50 50<br>0 50 50<br>0 50 90<br>0 50 50 | 50 50 0<br>50 90 0<br>50 50 0<br>0 0 1<br>0 1 1<br>0 0 1<br>Total de pixels defeituosos: 4 | $dx=1$ desloca a leitura uma coluna à direita; a última coluna de `alin` fica sem correspondência <br> (vira 0) e é sistematicamente marcada; o defeito real (90) também é detectado. |
| 2<br>2<br>0 0<br>20<br>10 10<br>10 10<br>10 10<br>10 60 | 10 10<br>10 60<br>0 0<br>0 1<br>Total de pixels defeituosos: 1 | Sem deslocamento ($dx=dy=0$): `alin` é idêntica a `cap`; apenas o defeito real (60) é detectado. |


In [124]:
#| label: fig-06-sim-ep07
#| fig-cap: "Simulador: *Pipeline* de Inspeção — Registro por Translação e Subtração"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0607" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Registro por Translação + Subtração</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟣 |ref − alin(dx,dy)| &gt; T</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste o deslocamento da esteira (dx) e o limiar T. Observe como a borda "fantasma" some quando dx=0.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:4px;">
        <label style="font-size:12px;font-weight:bold;color:#6d28d9;">Deslocamento horizontal (dx)</label>
        <span id="ep0607_vl_dx" style="font-family:monospace;font-weight:bold;color:#6d28d9;">1</span>
      </div>
      <input id="ep0607_sl_dx" style="width:100%;accent-color:#6d28d9;" max="2" min="-2" step="1" type="range" value="1">
      <div style="display:flex;justify-content:space-between;margin:10px 0 4px;">
        <label style="font-size:12px;font-weight:bold;color:#6d28d9;">Limiar T</label>
        <span id="ep0607_vl_t" style="font-family:monospace;font-weight:bold;color:#6d28d9;">30</span>
      </div>
      <input id="ep0607_sl_t" style="width:100%;accent-color:#6d28d9;" max="100" min="0" step="5" type="range" value="30">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:14px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;">ref</p>
        <div id="ep0607_g_ref" style="display:grid;grid-template-columns:repeat(3,40px);gap:3px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;">alin (registrada)</p>
        <div id="ep0607_g_alin" style="display:grid;grid-template-columns:repeat(3,40px);gap:3px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;">máscara</p>
        <div id="ep0607_g_mask" style="display:grid;grid-template-columns:repeat(3,40px);gap:3px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0607_debug" style="margin-top:16px;background:#ede9fe;border-radius:8px;padding:10px;border:1px solid #c4b5fd;font-family:monospace;font-size:11px;color:#5b21b6;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var N = 3;
    var ref = [[50,50,50],[50,50,50],[50,50,50]];
    // cap representa a peça já deslocada 1 px à direita (col 0 = 0) mais um defeito em (1,2)
    var cap = [[0,50,50],[0,50,90],[0,50,50]];

    var slDx = root.querySelector('#ep0607_sl_dx');
    var vlDx = root.querySelector('#ep0607_vl_dx');
    var slT  = root.querySelector('#ep0607_sl_t');
    var vlT  = root.querySelector('#ep0607_vl_t');
    var gRef = root.querySelector('#ep0607_g_ref');
    var gAlin= root.querySelector('#ep0607_g_alin');
    var gMask= root.querySelector('#ep0607_g_mask');
    var dbg  = root.querySelector('#ep0607_debug');

    function cellStyle(g, w, extra){
      return 'width:40px;height:40px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:10px;font-weight:bold;font-family:monospace;border:1px solid #ccc;' +
        'background:rgb('+g+','+g+','+g+');color:'+(g>140?'#000':'#fff')+';' + (extra||'');
    }

    function render(){
      var dx = parseInt(slDx.value);
      var T  = parseInt(slT.value);
      vlDx.textContent = dx;
      vlT.textContent = T;
      gRef.innerHTML=''; gAlin.innerHTML=''; gMask.innerHTML='';
      var alin = [], mask = [], total = 0;
      for(var i=0;i<N;i++){
        var rowA=[], rowM=[];
        for(var j=0;j<N;j++){
          var pj = j + dx;
          var v = (pj>=0 && pj<N) ? cap[i][pj] : 0;
          rowA.push(v);
          var diff = Math.abs(ref[i][j] - v);
          var m = diff > T ? 1 : 0;
          if(m) total++;
          rowM.push(m);
        }
        alin.push(rowA); mask.push(rowM);
      }
      for(var i=0;i<N;i++){
        for(var j=0;j<N;j++){
          var cr = document.createElement('div'); cr.style.cssText = cellStyle(ref[i][j]); cr.textContent = ref[i][j];
          gRef.appendChild(cr);
          var ca = document.createElement('div'); ca.style.cssText = cellStyle(alin[i][j]); ca.textContent = alin[i][j];
          gAlin.appendChild(ca);
          var m = mask[i][j];
          var cm = document.createElement('div');
          cm.style.cssText = 'width:40px;height:40px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:11px;font-weight:bold;font-family:monospace;' +
            (m ? 'background:#fecaca;color:#7f1d1d;border:1px solid #f87171;' : 'background:#f3f4f6;color:#9ca3af;border:1px solid #e5e7eb;');
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }
      dbg.textContent = 'dx=' + dx + '  |  T=' + T + '  |  Total de pixels defeituosos: ' + total + '/9';
    }
    slDx.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0607');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


In [125]:
%%writefile EP06_07.py
# Código Python

Overwriting EP06_07.py


In [126]:
TestSuite("EP06_07.py").run()